## Libraries Importing

In [2]:
import pandas as pd

# from google.colab import drive
# drive.mount('/content/drive')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier

import joblib

# -----------------------------------------------------------------------------------

## Get dataset

In [6]:
# path:str = "/content/drive/MyDrive/machine_break_prediction/data/processed/factory_sensor_processed.csv"
path:str="../data/processed/factory_sensor_processed.csv"
def get_data(path:str = path) -> pd.DataFrame:
    df = pd.read_csv(path, engine='pyarrow' )
    return df

df = get_data()
df.head()

,Installation_Year,Temperature_C,Vibration_mms,Sound_dB,Oil_Level_pct,Coolant_Level_pct,Power_Consumption_kW,Last_Maintenance_Days_Ago,Maintenance_History_Count,Failure_History_Count,...,Machine_Type_Pick_and_Place,Machine_Type_Press_Brake,Machine_Type_Pump,Machine_Type_Robot_Arm,Machine_Type_Shrink_Wrapper,Machine_Type_Shuttle_System,Machine_Type_Vacuum_Packer,Machine_Type_Valve_Controller,Machine_Type_Vision_System,Machine_Type_XRay_Inspector
0,2027.0,73.43,12.78,83.72,36.76,68.74,84.95,153.0,4.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2032.0,58.32,14.99,77.04,100.00,62.13,154.61,136.0,5.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2003.0,49.63,23.78,69.08,42.96,35.96,51.90,258.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2007.0,63.73,12.38,85.58,94.90,48.94,75.61,43.0,4.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,2016.0,42.77,4.42,96.72,47.56,53.78,224.93,346.0,4.0,2.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


# -----------------------------------------------------------------------------------

## Train and Test split

In [31]:
x = df.drop(columns=['Failure_Within_7_Days'])
y = df['Failure_Within_7_Days']

In [32]:
X_train,X_test,Y_train,Y_test= train_test_split(x,y,random_state=42,test_size=0.2)

# -----------------------------------------------------------------------------------

## OverSampling

In [33]:
smote = SMOTE(random_state=42)
X_train, Y_train = smote.fit_resample(X_train, Y_train)

# -----------------------------------------------------------------------------------

#### Fast comparison between train and test data

In [52]:
def evaluate_model(*,model,method = 'test') -> None:
    """
    evaluating model using 4 metrices:
    - Recall
    - Precision
    - f1_score
    - Accuracy

    please note that the function
    can be used on train data
    by simple argument " method =='train' " inside the function
    """
    print("--------------------------")
    print(f'Classification Report for {method} data:')

    if method == 'test':
        Y_pred = model.predict(X_test)
        print(classification_report(Y_test, Y_pred))
        print(f"Overall Accuracy: {accuracy_score(Y_test, Y_pred)}")
    else:
        Y_pred = model.predict(X_train)
        print(classification_report(Y_train, Y_pred))
        print(f"Overall Accuracy: {accuracy_score(Y_train, Y_pred)}")
    print("--------------------------")


# -----------------------------------------------------------------------------------

## Model Training

- random forest

In [44]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    n_jobs=-1,
    random_state=42
)

In [45]:
rf_model.fit(X_train, Y_train)

RandomForestClassifier(max_depth=15, n_jobs=-1, random_state=42)

In [51]:
evaluate_model(model=rf_model, method='train')

--------------------------
        Classification Report for train data:
              precision    recall  f1-score   support

       False       1.00      0.94      0.97    140237
        True       0.95      1.00      0.97    140237

    accuracy                           0.97    280474
   macro avg       0.97      0.97      0.97    280474
weighted avg       0.97      0.97      0.97    280474

Overall Accuracy: 0.9691450900974778
--------------------------


In [53]:
evaluate_model(model=rf_model)

--------------------------
Classification Report for test data:
              precision    recall  f1-score   support

       False       1.00      0.94      0.96     35032
        True       0.48      0.94      0.64      2257

    accuracy                           0.94     37289
   macro avg       0.74      0.94      0.80     37289
weighted avg       0.96      0.94      0.95     37289

Overall Accuracy: 0.9359060312692751
--------------------------


# -----------------------------------------------------------------------------------

- XgBoost

In [40]:
xgboost_model = XGBClassifier(
   n_estimators=200, learning_rate=0.05, max_depth=5
)

In [41]:
xgboost_model.fit(X_train, Y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [42]:
evaluate_model(model=xgboost_model, method='train')

--------------------------
Classification Report for train data:
              precision    recall  f1-score   support

       False       0.99      0.95      0.97    140237
        True       0.95      0.99      0.97    140237

    accuracy                           0.97    280474
   macro avg       0.97      0.97      0.97    280474
weighted avg       0.97      0.97      0.97    280474

Overall Accuracy: 0.9725215171459743
--------------------------


In [43]:
evaluate_model(model=xgboost_model)

--------------------------
Classification Report for test data:
              precision    recall  f1-score   support

       False       0.99      0.95      0.97     35032
        True       0.54      0.90      0.67      2257

    accuracy                           0.95     37289
   macro avg       0.76      0.92      0.82     37289
weighted avg       0.97      0.95      0.95     37289

Overall Accuracy: 0.9468207782455952
--------------------------


In [ ]:
joblib.dump(rf_model, '../models/model.pkl')
joblib.dump(X_train.columns.tolist(), '../models/model_columns.pkl')
print("Model saved!")

Model saved!
